# 🔥🔥 Let's Make PubMed abstracts easier to read !! with : " Brief-Lit " 📄 🔥🔥


---



##*Project description :*

## Start :

Confirming GPU

In [1]:
!nvidia-smi -L

/bin/bash: line 1: nvidia-smi: command not found


## All Imports

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import itertools
from tensorflow.keras.layers import TextVectorization
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score  , accuracy_score
import tensorflow_hub as hub
import tf_keras

## Getting the Data

**PubMed 200k RCT dataset**

The PubMed 200k RCT dataset is described in Franck Dernoncourt, Ji Young Lee. PubMed 200k RCT: a Dataset for Sequential Sentence Classification in Medical Abstracts. International Joint Conference on Natural Language Processing (IJCNLP). 2017.

**[Abstract:](https://)**

PubMed 200k RCT is new dataset based on PubMed for sequential sentence classification. The dataset consists of approximately 200,000 abstracts of randomized controlled trials, totaling 2.3 million sentences. Each sentence of each abstract is labeled with their role in the abstract using one of the following classes: background, objective, method, result, or conclusion. The purpose of releasing this dataset is twofold. First, the majority of datasets for sequential short-text classification (i.e., classification of short texts that appear in sequences) are small: we hope that releasing a new large dataset will help develop more accurate algorithms for this task. Second, from an application perspective, researchers need better tools to efficiently skim through the literature. Automatically classifying each sentence in an abstract would help researchers read abstracts more efficiently, especially in fields where abstracts may be long, such as the medical field.

In [3]:
!git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git

Cloning into 'pubmed-rct'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 39 (delta 8), reused 5 (delta 5), pack-reused 25 (from 1)
Receiving objects: 100% (39/39), 177.08 MiB | 15.57 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (13/13), done.


In [4]:
!ls pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign

dev.txt  test.txt  train.txt


In [5]:
data_dir = "/content/pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign"
filenames = [data_dir + filename for filename in os.listdir(data_dir)]
filenames

['/content/pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_signdev.txt',
 '/content/pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_signtest.txt',
 '/content/pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_signtrain.txt']

## Preprocess Data

In [6]:
# Reading The Lines  Of The Document

def get_lines(filename):
  with open(filename , "r") as f :
    return f.readlines()

In [7]:
train_lines = get_lines(data_dir+"/train.txt")
train_lines[:10]

['###24293578\n',
 'OBJECTIVE\tTo investigate the efficacy of @ weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at @ weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .\n',
 'METHODS\tA total of @ patients with primary knee OA were randomized @:@ ; @ received @ mg/day of prednisolone and @ received placebo for @ weeks .\n',
 'METHODS\tOutcome measures included pain reduction and improvement in function scores and systemic inflammation markers .\n',
 'METHODS\tPain was assessed using the visual analog pain scale ( @-@ mm ) .\n',
 'METHODS\tSecondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis Index scores , patient global assessment ( PGA ) of the severity of knee OA , and @-min walk distance ( @MWD ) .\n',
 'METHODS\tSerum levels of interleukin @ ( IL-@ ) , IL-@ , tumor necrosis factor ( TNF ) - , and 

In [8]:
len(train_lines)

210040

In [9]:
def preprocess_text_with_line_numbers(filename):
  input_lines = get_lines(filename)
  abstract_lines = ""
  abstract_samples = []

  for line in input_lines:
    if line.startswith("###"):
      abstract_id = line
      abstract_lines = ""
    elif line.isspace():
      abstract_line_split = abstract_lines.splitlines()


      for abstract_line_number, abstract_line in enumerate(abstract_line_split):
        line_data = {}
        target_text_split = abstract_line.split("\t")
        line_data["target"] = target_text_split[0]
        line_data["text"] = target_text_split[1].lower()
        line_data["line_number"] = abstract_line_number
        line_data["total_lines"] = len(abstract_line_split) - 1
        abstract_samples.append(line_data)
    else:
      abstract_lines += line
  return abstract_samples

In [10]:
train_samples = preprocess_text_with_line_numbers(data_dir + "/train.txt")
val_samples = preprocess_text_with_line_numbers(data_dir + "/dev.txt")
test_samples = preprocess_text_with_line_numbers(data_dir + "/test.txt")

In [11]:
print("Length Of Training Samples : " , len(train_samples))
print("Length Of Validation Samples : " , len(val_samples))
print("Length Of Test Samples : " , len(test_samples))

Length Of Training Samples :  180040
Length Of Validation Samples :  30212
Length Of Test Samples :  30135


In [12]:
train_samples[:10]

[{'target': 'OBJECTIVE',
  'text': 'to investigate the efficacy of @ weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at @ weeks in older adults with moderate to severe knee osteoarthritis ( oa ) .',
  'line_number': 0,
  'total_lines': 11},
 {'target': 'METHODS',
  'text': 'a total of @ patients with primary knee oa were randomized @:@ ; @ received @ mg/day of prednisolone and @ received placebo for @ weeks .',
  'line_number': 1,
  'total_lines': 11},
 {'target': 'METHODS',
  'text': 'outcome measures included pain reduction and improvement in function scores and systemic inflammation markers .',
  'line_number': 2,
  'total_lines': 11},
 {'target': 'METHODS',
  'text': 'pain was assessed using the visual analog pain scale ( @-@ mm ) .',
  'line_number': 3,
  'total_lines': 11},
 {'target': 'METHODS',
  'text': 'secondary outcome measures included the western ontari

In [ ]:
train_df = pd.DataFrame(train_samples)
val_df = pd.DataFrame(val_samples)
test_df = pd.DataFrame(test_samples)
train_df.head(14)

In [ ]:
train_df["target"].value_counts()

In [ ]:
# Let's check the length of Different lines
train_df["total_lines"].plot.hist();

In [ ]:
train_sentences = train_df["text"].tolist()
test_sentences = test_df["text"].tolist()
val_sentences = val_df["text"].tolist()

len(train_sentences) , len(test_sentences) , len(val_sentences)

Transforming Numeric Labels :

One Hot Encoding

In [ ]:
ohe_encoder = OneHotEncoder(sparse_output=False)
train_labels_one_hot = ohe_encoder.fit_transform(train_df["target"].to_numpy().reshape(-1 , 1))
test_labels_one_hot = ohe_encoder.transform(test_df["target"].to_numpy().reshape(-1 , 1))
val_labels_one_hot = ohe_encoder.transform(val_df["target"].to_numpy().reshape(-1 , 1))

# Note : The fit_transform method of OneHotEncoder expects a 2D array as input. Reshaping the target column from a 1D array (single column) to a 2D array (with one row and many columns) allows the encoder to function properly. This is because it treats each sample as a row and each category as a column when creating the one-hot encoding.

Label Encoding

In [ ]:
label_encoder = LabelEncoder()
train_labels_encoder = label_encoder.fit_transform(train_df["target"].to_numpy().reshape(-1 , 1))
test_labels_encoder = label_encoder.transform(test_df["target"].to_numpy().reshape(-1 , 1))
val_labels_encoder = label_encoder.transform(val_df["target"].to_numpy().reshape(-1 , 1))
train_labels_encoder

In [ ]:
def evaluation_metrics_tf(model , test_pred , test_labels) :
    accuracy = tf.keras.metrics.Accuracy()
    precision = tf.keras.metrics.Precision()
    recall = tf.keras.metrics.Recall()

    # Updating state
    accuracy.update_state(test_pred , test_labels)
    precision.update_state(test_pred , test_labels)
    recall.update_state(test_pred , test_labels)

    # Evaluate Scores
    accuracy_score = accuracy.result().numpy()
    precision_score = precision.result().numpy()
    recall_score = recall.result().numpy()
    f1_score = 2 * (precision_score * recall_score) / (precision_score + recall_score + 1e-7)
    confusion_matrix_tf = tf.math.confusion_matrix(test_labels , test_pred)

    # Results Disctionary
    results = {
        "Accuracy" : accuracy_score ,
        "Recall" : recall_score ,
        "Precision" : precision_score ,
        "F1-Score" : f1_score ,
        "Confusion Matrix" : confusion_matrix_tf
    }

    return results

## Experimentation Of Different Models :

### Model 0 : Getting a baseline model : " Machine Learing Based => Multinomial NB"

In [ ]:
model_0 = Pipeline([
    ("tfid", TfidfVectorizer()),
    ("clf", MultinomialNB())
]
)


model_0.fit(train_sentences , train_labels_encoder)

In [ ]:
model_0.score(X= val_sentences ,
                 y= val_labels_encoder)

In [ ]:
model_0_preds = model_0.predict(val_sentences)
model_0_preds

In [ ]:
def evaluation_metrics_sklearn(model , test_pred , test_labels) :
    accuracy = accuracy_score(test_labels , test_pred)
    precision = precision_score(test_labels, test_pred, average='weighted')
    recall = recall_score(test_labels , test_pred, average='weighted')
    f1= f1_score(test_labels , test_pred, average='weighted')
    confusion = confusion_matrix(test_labels , test_pred)
    results = {
        "Accuracy" : accuracy ,
        "Recall" : recall ,
        "Precision" : precision ,
        "F1-Score" : f1 ,
        "Confusion Matrix" : confusion
    }
    return results

In [ ]:
model_0_results = evaluation_metrics_sklearn(test_labels= val_labels_encoder ,
                                    test_pred= model_0_preds , model=model_0)
model_0_results

In [ ]:
# from helper_functions import calculate_results
# model_0_results = calculate_results(y_true= val_labels_encoder ,
#                                     y_pred= model_0_preds)
# model_0_results

### Model 1 : " Conv 1D  "

In [ ]:
# Lengths of Each Sentences
sentence_len_list = [len(sentence.split()) for sentence in train_sentences]
max_len = int(np.max(sentence_len_list))
min_len = int(np.min(sentence_len_list))
avg_len = int(np.mean(sentence_len_list))
_95_percentile_len = int(np.percentile(sentence_len_list , 95))
avg_len , max_len , min_len , _95_percentile_len

In [ ]:
# Distribution of the lengths of the sentences
plt.hist(sentence_len_list , bins = 20)

In [ ]:
# Vocab length
sentences_list = [sentence.split() for sentence in train_sentences]
word_list = list(itertools.chain.from_iterable(sentences_list))
vocab_len = len(set(word_list))
vocab_len

#### Creating the Text Vectorizer

In [ ]:
text_vectorizer = TextVectorization(max_tokens = vocab_len ,
                                    output_sequence_length = _95_percentile_len)

text_vectorizer.adapt(train_sentences)

In [ ]:
vocab = text_vectorizer.get_vocabulary()
print(f"Length of vocab : {len(vocab)}")

In [ ]:
text_vectorizer.get_config()

 #### Creating embedding Layer

In [ ]:
embedding_layer = layers.Embedding(input_dim=len(vocab),
                                   output_dim = 512 ,
                                   mask_zero = True ,
                                   name = "embedding_layer",
                                   )

In [ ]:
# Checking a sample Vectorized and embedded sentence

sample_sentence = train_sentences[np.random.randint(0 , len(train_sentences))]
sample_sentence_vectorized = text_vectorizer([sample_sentence])
sample_sentence_embedded = embedding_layer(sample_sentence_vectorized)
print(f"Sample Sentence : {sample_sentence}")
print(f"Sample Sentence Vectorized : {sample_sentence_vectorized}")
print(f"Sample Sentence Embedded : {sample_sentence_embedded}")

Turning Our Datasets  Into TensorFlow Datasets:  "TensorFlow tf.data API"

In [ ]:
train_tf_dataset = tf.data.Dataset.from_tensor_slices((train_sentences , train_labels_one_hot))
val_tf_dataset = tf.data.Dataset.from_tensor_slices((val_sentences , val_labels_one_hot))
test_tf_dataset = tf.data.Dataset.from_tensor_slices((test_sentences , test_labels_one_hot))

In [ ]:
# Prefected Datasets
train_tf_dataset= train_tf_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
val_tf_dataset = val_tf_dataset.batch(32).prefetch(tf.data.AUTOTUNE)
test_tf_dataset = test_tf_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_tf_dataset

In [ ]:
# Model Building
inputs = layers.Input(shape=(1,) , dtype=tf.string)
vectorized_text = text_vectorizer(inputs)
embedding_text = embedding_layer(vectorized_text)
x = layers.Conv1D(128 , kernel_size = 5 , padding="same" , activation ="relu")(embedding_text)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(train_df['target'].nunique() , activation="softmax")(x)
model_1 = tf.keras.Model(inputs , outputs)

In [ ]:
# Compile Model
model_1.compile(loss="categorical_crossentropy" ,
                optimizer = tf.keras.optimizers.Adam() ,
                metrics = ["accuracy"])

In [ ]:
model_1.summary()

In [ ]:
# Fitting the model_1
model_1_history = model_1.fit(train_tf_dataset ,steps_per_epoch=int(0.1*len(train_tf_dataset)) ,  epochs = 5 , validation_data = val_tf_dataset , validation_steps = int(0.1*len(val_tf_dataset)))

In [ ]:
model_1.evaluate(val_tf_dataset)

In [ ]:
model_1_pred_probs = model_1.predict(val_tf_dataset)
model_1_pred_probs

In [ ]:
model_1_preds = tf.argmax(model_1_pred_probs , axis=1)
model_1_preds

In [ ]:
model_1_results = evaluation_metrics_tf(model= model_1 ,test_labels=val_labels_encoder , test_pred=model_1_preds)
model_1_results

### Model 2 : " Using Pretrained Token Embeddings  "

In [ ]:
# We can use this encoding layer in place of our text_vectorizer and embedding layer

sentence_encoder_layer = hub.KerasLayer("https://tfhub.dev/google/universal-sentence-encoder/4",
                                        input_shape=[],
                                        dtype=tf.string,
                                        trainable=False,
                                        name="USE")


In [ ]:
sample_sentence = train_sentences[np.random.randint(0 , len(train_sentences))]
sample_sentence_encoded_USE = sentence_encoder_layer([sample_sentence])
print(f"Sample Sentence Embedded : {sample_sentence_encoded_USE}")

In [ ]:
# !pip install tensorflow==2.14 tensorflow-hub==0.15

In [ ]:
# Creating the model
inputs = layers.Input(shape=() , dtype=tf.string)
embedding_layer_USE = sentence_encoder_layer(inputs)
x = layers.Dense(128 , activation="relu")(embedding_layer_USE)
outputs = layers.Dense(train_df['target'].nunique() , activation="softmax")(x)
model_2 = tf.keras.Model(inputs ,
                         outputs,
                         name = "model_2_USE")

In [ ]:
model_2.summary()